# Обучение одиночного нейрона

`Neuron` самостоятельно хранит веса и смещение, выполняет прямой и
обратный проход и обновляет параметры. Это отдельная модель для
изучения одного нейрона. `Unit` выполняет другую роль: он показывает
один нейрон, параметры которого принадлежат матричному `Layer`.

На примере AND сначала вычислим градиенты вручную по шагам, затем
запустим цикл обучения и сравним ответы с таблицей истинности.

In [31]:
import numpy as np

from open_neuron.activations import sigmoid
from open_neuron.losses import MSE
from open_neuron.neurons import Neuron
from open_neuron.optimizers import SGD

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [0], [0], [1]])

neuron = Neuron(
    weights=[0.05, -0.12],
    bias=1.0,
    activation=sigmoid,
    loss=MSE(),
    optimizer=SGD(learning_rate=0.7, momentum=0.3),
)

## Прямой и обратный проход

Для каждого примера нейрон вычисляет взвешенную сумму и применяет
сигмоиду. Затем MSE превращает остаток `A - y` в градиент потерь;
производная сигмоиды передаёт его к весам и смещению. Вызов
`backward` только вычисляет градиенты — параметры изменит `update`,
который вызывается циклом `fit`.

In [32]:
initial_predictions = neuron.forward(X)
initial_loss = neuron.loss_function.evaluate_loss(y, initial_predictions)
error = neuron.loss_function.evaluate_error(y, initial_predictions)
neuron.backward(error)

print('Взвешенные суммы Z:', neuron.Z.ravel())
print('Выходы A:', initial_predictions.ravel())
print('Начальная MSE:', initial_loss)
print('Градиент весов:', neuron.gradients['weights'].ravel())
print('Градиент смещения:', neuron.gradients['bias'].ravel())

Взвешенные суммы Z: [1.   0.88 1.05 0.93]
Выходы A: [0.73105858 0.70682222 0.7407749  0.71707529]
Начальная MSE: 0.41570953573934516
Градиент весов: [0.04242491 0.04453582]
Градиент смещения: [0.1875278]


## Обучение и предсказание

`fit` повторяет прямой проход, вычисление градиентов и обновление
параметров. После обучения `predict` считает выходы без обновления.
Округление вероятностей до ближайшего целого показывает предсказанный
класс для этой задачи.

In [33]:
neuron.fit(X, y, epochs=10, shuffle=False)
predictions = neuron.predict(X)

print('Вероятности:', predictions.ravel())
print('Предсказанные классы:', np.round(predictions).astype(int).ravel())
print('Ожидаемые классы:    ', y.ravel())
print('Состояние нейрона после обучения:')
print(neuron)

Training started with Overall Loss 0.4157 on Train Data
Epoch 1/10, Loss: 0.4291, Validation Loss: 0.2981 on Train Data
Epoch 2/10, Loss: 0.3310, Validation Loss: 0.2353 on Train Data
Epoch 3/10, Loss: 0.2675, Validation Loss: 0.2043 on Train Data
Epoch 4/10, Loss: 0.2331, Validation Loss: 0.1848 on Train Data
Epoch 5/10, Loss: 0.2116, Validation Loss: 0.1692 on Train Data
Epoch 6/10, Loss: 0.1953, Validation Loss: 0.1556 on Train Data
Epoch 7/10, Loss: 0.1814, Validation Loss: 0.1437 on Train Data
Epoch 8/10, Loss: 0.1692, Validation Loss: 0.1333 on Train Data
Epoch 9/10, Loss: 0.1581, Validation Loss: 0.1243 on Train Data
Epoch 10/10, Loss: 0.1481, Validation Loss: 0.1164 on Train Data
Training completed with Overall Loss 0.1164
Вероятности: [0.18204995 0.33930897 0.36972751 0.57511684]
Предсказанные классы: [0 0 0 1]
Ожидаемые классы:     [0 0 0 1]
Состояние нейрона после обучения:
Neuron 11, inputs: [1 1], weights: [0.9691 0.8361], bias: [-1.5025], z: 0.3027590065330077, activation